# Data Extraction
This stage involves 2 components:
- Linguistic feature extraction (Will be done in RStudio by using R - **02_PPA_linguistic_extraction**)
- Acoustic feature extraction (Will be done in this Colab Notebook - **03_PPA_acoustic_extraction.ipynb**)

## Overview of Acoustic Feature Extraction

**Libraries used to extract acoustic features**
- `praat-parselmouth` - v0.4.7
- `librosa` - v0.11.0

**WHY**: These libraries provide complimentary functions for analyzing different aspects of speech.

**1. Features extracted using `praat-parselmouth`**

`praat-parselmouth` offers precise and detailed analysis of speech making it effective to extract pitch, voice quality, and formant measurements. It focuses on the specifics, and frame-by-frame analyses that complement `librosa` broad analyses.

-   **Pitch-related features:** Fundamental frequency (F0) statistics (mean, minimum, maximum, standard deviation).
    *   **Fundamental Frequency (F0):** Is a objective, physical measurement in Hertz (Hz) the rate at which a person's vocal cords vibrate. It's the same concept as pitch which is subjective, and measures how high a low a sound is perceived in the human ear and brain
        * Mean conveys average
        * Minimum and Maximum convey the typical range
        * Standard Deviation conveys pitch variability
-   **Voice Quality features:** Jitter and Shimmer measure irregularities with vocal fold vibration.
    *   **Jitter:** Measures irregular variations in the period (time it takes a wave to start and stop at x-axis) of a single wave
    *   **Shimmer:** Measures irregular variations in the amplitude (loudness measured from x-axis to crest or trough) of a single audio wave
-   **Prosodic features:** Speaking rate and intensity statistics measure voice quality
    * **Speaking Rate:** Measures syllables spoken divided by total duration (active + pauses) to produce speech fluency
    * **Intensity Statistics:** Measure loudness (amplitude) of sound to indicat how strong or soft a voice is. Analyzing intensity's mean, min, max, and standard deviation helps understand typical range, and variance as mentioned for fundamental frequency.
-   **Formant features (F1, F2, F3):** Specific frequency peaks influenced by vocal tract shape and are crucial for distinguishing different vowel sounds.
      * **F1** influenced by tongue height and jaw opening
      * **F2** is influenced by tongue position (frontness/backness)
      * **F3** influenced by lip rounding and specific modifications.

**2. Features extracted using `librosa`**

`librosa` offers efficient audio analysis making it ideal for extracting features like MFCCs, spectral properties, and time-domain energy from a musical and general audio processing perspective. It can process datasets to extract broader acoustic features that complement `praat-parselmouth` detailed analyses.

-   **Time Domain features:** Capture how characteristics change over time.
    *   **Zero Crossing Rate (ZCR):** Measures how many times the audio wave crosses the x-axis.
    *   **Root Mean Square (RMS) Energy:** Measures intensity/loudness of the audio signal over time.

- **Spectral features:** Provide insights into how acoustic energy is distributed across frequencies, and texture/identity of voice.
    *   **Mel-frequency Cepstral Coefficients (MFCCs):** Set of features that mimic how the human ear perceives different frequencies. Characterize the unique texture and identity of voice.
    *   **Spectral Centroid:** Indicates the center/average of sound's spectrum. Conveys if sound is base heavy or bright.
    *   **Spectral Spread:** Measures frequency variance around the spectral centroid.
    *   **Spectral Rolloff:** Contains frequency below a certain threshold (e.g., 85%) to distinguish between low and high frequency sounds, especially outliers.
    *   **Chroma Feature:** These features represent the 12 different pitch classes (like the 12 notes in a musical octave) which help capture harmony and melody to see voice has music intonation or sounds flat.

This combination will gather a rich set of acoustic data for PPA Variant classification.

### Dataset 1: Depaul
2 .cha files, 2 .mp3/.mp4 files, 2 .wav files

#### Step 1: Converting `.mp3` files to `.wav` files by using `pydub` library

This is done to ensure consistent processing and compatibility with `parselmouth` and `librosa`.

In [ ]:
# Install, Import and Note Version of Pydub
print(f"Installing Pydub")
!pip install pydub
import pydub
print(f"Successfully imported Pydub")
# pydub version is 0.25.1 and the print function doesn't work as pydub doesn't store its version in a _version_ atrribute like other libraries

In [ ]:
# 1. Import required libraries
# Install if needed
import os # Used for navigating folders, creating directories, and finding folders
import re # re stands for Regular Expressions which is a tool that searches for specific patterns (like timestamps) in text files
from google.colab import drive # Lets python read and write files directly inside google drive
from pydub import AudioSegment # AudioSegment is the audio engine in pydub library which converts audio files (.mp3/.mp4 to .wav)

# 2. Connect Drive and Colab so files can be accessed
drive.mount('/content/drive')

# 3. Establishing file paths
# Shows where the .cha and .mp3/.mp4 files are
root_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/depaul/depaul_raw'
# Shows where .wav files should be store
output_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/depaul/depaul_wav'

os.makedirs(output_folder, exist_ok=True) # Creates the output_folder if it doesn't already exist to prevent code from crashing

# 4. Writing the function to extract millisecond timestamps for *PAR:
def extract_par_timestamps(cha_file_path): # Takes one .cha file at a time
    timestamps = [] # Make an empty list to store the start_time and end_time millisecond pairs
    timestamp_pattern = re.compile(r'[\x15\u0015](\d+)_(\d+)[\x15\u0015]') # Tells Python the search rule to find timestamp numbers in DementiaBank CHAT Transcripts

    with open(cha_file_path, 'r', encoding='utf-8', errors='ignore') as file: # Opens .cha text transcript line by line
        for line in file: # Loops through every line of text in the transcript
            if line.startswith('*PAR:'): # Only process lines where the Participant (*PAR:) is speaking and ignores lines with *INV:
                match = timestamp_pattern.search(line) # Search a specific *PAR: line for timestamps and then move to other *PAR: line
                if match:
                    start_ms = int(match.group(1)) # Converts the start timestamp text into real integer numbers
                    end_ms = int(match.group(2)) # Converts the end timestamp text into real integer numbers
                    timestamps.append((start_ms, end_ms)) # Organizes the timestamps by listing start and end times
    return timestamps # Returns the complete list of patient speaking intervals back to the main program

# 5. Scans folders to find audio and text files
if os.path.exists(root_input_folder): # Checks if depaul_raw (file folder) exists in google drive so code doesn't crash
    found_media_count = 0 # Tracks how many audio files get found - Starts with 0

    for current_dir, subdirs, files in os.walk(root_input_folder): # Tells os.walk (a part of os library) to check the main folder and EVERY subfolder inside it just in case large files need organization and prevents code crash

        file_map = {f.lower(): f for f in files} # Builds a dictionary of file names so Python doesn't get confused by uppercase and lowercase extensions

        for filename in files: # Loops through each file found in the directory

         # Splits the file name into its extension (.mp3) and ID
            ext = os.path.splitext(filename)[1].lower()  # Gets ONLY the extension (.mp3) so ext can trigger file conversion
            base_name = os.path.splitext(filename)[0] # Gets ONLY the patient ID so exact .cha file can be matched


            if ext in ['.mp3', '.mp4', '.wav', '.m4a']: # Checks if file is a recognized audio/video file by utilizing seperated extension
                found_media_count += 1 # found_media_count (0) + 1 counts how many audio/video files it found
                print(f"Processing: {os.path.join(current_dir, filename)}")

                cha_filename = base_name + '.cha' # Seperated Patient ID + .cha to know the full file name of .cha files
                cha_path = os.path.join(current_dir, cha_filename) # Tells python where that specific .cha file is

                if os.path.exists(cha_path): # Confirms if matching .cha file exists
                    print(f"  Found matching .cha file: {cha_path}")
                    timestamps = extract_par_timestamps(cha_path) # Pulls all *PAR: timestamps out of the .cha file according to the function in part 4

                    if timestamps:
                        try: # Opens a safety block so one corrupted audio file won't crash whole code
                            # Load the full audio file into memory with the correct extension as each ext is run because the extensions were seperated
                            if ext == '.mp3':
                                full_audio = AudioSegment.from_mp3(os.path.join(current_dir, filename))
                            elif ext == '.mp4' or ext == '.m4a':
                                full_audio = AudioSegment.from_file(os.path.join(current_dir, filename), format='m4a')
                            elif ext == '.wav':
                                full_audio = AudioSegment.from_wav(os.path.join(current_dir, filename))

                            patient_audio = AudioSegment.empty() # Creates blank audio space where only patient intervals are loaded

                            for start_ms, end_ms in timestamps:  # Concentrates on the timestamps *PAR: is speaking
                            # Ensures timestamps are within audio bounds
                                start_ms = max(0, start_ms)   # Check if audio is below 0
                                end_ms = min(len(full_audio), end_ms) # Check if audio above end time of audio file
                                if start_ms < end_ms: # Safety net to ensure start time is before end time
                                    patient_audio += full_audio[start_ms:end_ms] # Loads the participant speech into the blank audio space

                            if patient_audio.duration_seconds > 0: # Verifies we actually extracted valid audio

                                relative_path = os.path.relpath(current_dir, root_input_folder) # Finds folder branch name of depaul_raw (file folder)
                                output_subdir = os.path.join(output_folder, relative_path) # Glues the branch name onto output path so each wav file in same subfolder name as in the original folder
                                os.makedirs(output_subdir, exist_ok=True) # Creates the output folder if it doesn't already exist so code doesn't crash

                                output_wav_path = os.path.join(output_subdir, f"{base_name}_PAR_clean.wav") # Names the new .wav files
                                patient_audio.export(output_wav_path, format="wav") # Exported into drive

                               # Print helpful messages that convey if code is a success
                               # OR pinpoint where the warning occurred to target debugging at a specific aspect of code that's causing the issue
                                print(f"    SUCCESS: Clean patient WAV saved at: {output_wav_path}")
                            else:
                                print(f"    WARNING: No valid patient speech segments extracted for {filename}.")
                        except Exception as e:
                            print(f"    ERROR processing {filename}: {e}")
                    else:
                        print(f"    WARNING: No *PAR: timestamps found in {cha_filename}.")
                else:
                    print(f"  WARNING: No matching .cha file ({cha_filename}) found in {current_dir}.")

    if found_media_count == 0:
        print(f"No audio files (mp3, mp4, wav, m4a) found in {root_input_folder} or its subfolders.")
else:
    print(f"ERROR: Could not find the specified root input directory: {root_input_folder}")

print("\nBatch processing complete.")



#### Step 2: Acoustic Feature Extraction using `parselmouth`

This section will focus on extracting pitch, voice quality, prosodic, and formant features using the `parselmouth` library.

In [ ]:
# 1. Install and Import Required Libraries
import os                         # Used for navigating folders and building file paths
import numpy as np                # Calculates mean, min, max, and std
import pandas as pd               # Converts results into a clean data frame and exports CSV
from google.colab import drive    # Connects Google Drive to Google Colab
!pip install Praat-Parselmouth==0.4.7 # Install praat-parselmouth
import parselmouth                # Python library for Praat acoustic engine
from parselmouth.praat import call # Executes Praat C++ functions

# 2. Connect to Drive
drive.mount('/content/drive')

# 3. Establish File Path
# Shows where .wav files are stored
wav_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/depaul/depaul_wav'

# Shows where to store the csv file
output_csv_path = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/depaul/depaul_acoustic_features_parselmouth.csv'

# 4. Define feature extraction function
def extract_parselmouth_features(audio_file_path): # takes one single audio file path and extracts pitch, jitter, shimmer, intensity, speaking rate proxy, and formants

    features = {} # Creates an empty dictionary to store the features

    try:
        sound = parselmouth.Sound(audio_file_path) # Loads audio into Praat's memory
        total_duration = sound.get_total_duration() # Measures total duration of audio clip in seconds

        # Part A. Pitch (F0) Features (mean, min, max, std)
        pitch = sound.to_pitch() # Creates Parselmouth pitch object containing calculated frequencies and corresponding time stamps
        pitch_values = pitch.selected_array["frequency"] # Stores Frequency numbers in Hz as 1D Numpy array so it can be used to calculate F0 metrics
        pitch_values = pitch_values[pitch_values != 0] # Filter out silent sound frames because if a person spoke half the time and remained silent for half the time, keeping 0 cuts their average in half

        if len(pitch_values) > 0: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_f0'] = float(np.mean(pitch_values)) # Average
            features['min_f0'] = float(np.min(pitch_values)) # Lowest
            features['max_f0'] = float(np.max(pitch_values)) # Highest
            features['std_f0'] = float(np.std(pitch_values)) # Variation
        else: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_f0'] = 0.0
            features['min_f0'] = 0.0
            features['max_f0'] = 0.0
            features['std_f0'] = 0.0

        # Part B. Voice Quality (Jitter and Shimmer)
        # Calls Praat's C++ to scan sound wave and locate vocal cord pulses between 75 Hz - 500 Hz which is the normal range of Hz
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
        # Asks praat to calculate jitter, multiples by 100 to get %, and saves it
        features['jitter'] = float(call(point_process, "Get jitter (local)", 0.0, 0.0, 0.0001, 0.02, 1.3) * 100.0)
        # Asks praat to calculate shimmer, multiples by 100 to get %, and saves it
        features['shimmer'] = float(call([sound, point_process], "Get shimmer (local)", 0.0, 0.0, 0.0001, 0.02, 1.3, 1.6) * 100.0)

        # Part C. Prosodic Features
        # Intensity Statistics (mean, min, max, std)
        intensity = sound.to_intensity() # Converts audio to intensity object containing sound pressure (loudness) over time
        intensity_values = intensity.values[0] # Extracts volume in decibels for every single step
        valid_intensity = intensity_values[intensity_values > 0] # Filter out zero value frames so silent background gaps don't drag down speaker's real voice volume

        if len(valid_intensity) > 0: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_intensity'] = float(np.mean(valid_intensity)) # Average
            features['min_intensity'] = float(np.min(valid_intensity)) # Lowest
            features['max_intensity'] = float(np.max(valid_intensity)) # Highest
            features['std_intensity'] = float(np.std(valid_intensity)) # Variation
        else: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_intensity'] = 0.0
            features['min_intensity'] = 0.0
            features['max_intensity'] = 0.0
            features['std_intensity'] = 0.0

        # Speaking Rate and Duration
        features['total_duration_sec'] = float(total_duration) # Stores total length of audio file as seconds (duration)

        # Using Praat TextGrid silences
        textgrid = call(sound, "To TextGrid (silences)", 100, 0.0, -25.0, 0.1, 0.1, "silent", "sounding") # Tells Praat to slice audio into intervals labeled either sounding or silent based on decibels drops below -25 dB to isolate active vocalization and pause time, and calculate Articulation Ratio
        num_intervals = call(textgrid, "Get number of intervals", 1) # Asks TextGrid object for total count of alternating speech and pause slices.
        speech_duration = 0.0 # Creates tracker variable starting at 0 to track active talking time

        for i in range(1, num_intervals + 1): # To loop through every interval slice from 1 to the total number of intervals
            label = call(textgrid, "Get label of interval", 1, i) # Checks the text tag of current slice to see if silent or sounding
            if label == "sounding": # If the slice is active speech....
            # Extract the exact start and end timestamp in seconds to know where that speech slice begins and ends.
                start = call(textgrid, "Get start time of interval", 1, i)
                end = call(textgrid, "Get end time of interval", 1, i)
                speech_duration += (end - start) # To get total active speech duration

        features['speech_duration_sec'] = float(speech_duration) # Saves the final duration into the dictionary

        # Speaking Rate Metric (Active Speech / Total Time):
        features['speaking_rate_proxy'] = float(speech_duration / total_duration) if total_duration > 0 else 0.0 # Divides active speech time over duration in seconds to calculate articulation ratio; defaults to 0 if duration is 0

        # Part D. Formants (F1, F2, F3)
        formant = sound.to_formant_burg(time_step=0.01, maximum_formant=5500, max_number_of_formants=5) # Runs Burg's linear prediction algorithm (subtracts the vocal cords vibrations so shape of mouth during speech/formant features can be extracted) to track vocal tract movements up to 5500 Hz every 0.01 seconds.
        f1_vals, f2_vals, f3_vals = [], [], [] # Creates 3 empty lists to store three metrics

        for t in formant.ts(): # Loops through every time point formant measurements were taken
        # Fetches the specific frequency values (in Hz) for Formant 1, Formant 2, and Formant 3 at time t.
            f1 = formant.get_value_at_time(1, t)
            f2 = formant.get_value_at_time(2, t)
            f3 = formant.get_value_at_time(3, t)
# Checks if each value is missing a value (NaN) or is 0 and adds valid numbers to their lists
            if not np.isnan(f1) and f1 > 0: f1_vals.append(f1)
            if not np.isnan(f2) and f2 > 0: f2_vals.append(f2)
            if not np.isnan(f3) and f3 > 0: f3_vals.append(f3)
# Calculates average if more than 0 or sets to 0 if no formants detected
        features['mean_f1'] = float(np.mean(f1_vals)) if f1_vals else 0.0
        features['mean_f2'] = float(np.mean(f2_vals)) if f2_vals else 0.0
        features['mean_f3'] = float(np.mean(f3_vals)) if f3_vals else 0.0

    except Exception as e: # Catches file corruption error, prints error message, makes it 0, and keeps code running
        print(f"Error processing audio file: {e}")
        features = {
            'mean_f0': 0.0, 'min_f0': 0.0, 'max_f0': 0.0, 'std_f0': 0.0,
            'jitter': 0.0, 'shimmer': 0.0,
            'mean_intensity': 0.0, 'min_intensity': 0.0, 'max_intensity': 0.0, 'std_intensity': 0.0,
            'total_duration_sec': 0.0, 'speech_duration_sec': 0.0, 'speaking_rate_proxy': 0.0,
            'mean_f1': 0.0, 'mean_f2': 0.0, 'mean_f3': 0.0
        }

    return features


# 5. Feature Extraction Master List
all_extracted_rows = [] # Creates an empty file to capture all extracted features from .wav file

if os.path.exists(wav_input_folder): # Checks if .wav file folder exists
    for current_dir, subdirs, files in os.walk(wav_input_folder): # Checks the main folder and any subfolders inside it
        for filename in files:
            if filename.endswith('.wav'): # Isolates files that end with .wav extension

                # audio_file_path creation ***
                audio_file_path = os.path.join(current_dir, filename) # Joins the folder path and filename into complete file path string: audio_file_path

                print(f"Extracting features from: {filename}")

                # Calls the main function in step 4 to be executed and extract all acoustic features
                row_features = extract_parselmouth_features(audio_file_path)

                # Removes _PAR_clean.wav to create a clean participant ID, and stores the raw filename
                row_features['participant_id'] = filename.replace('_PAR_clean.wav', '')
                row_features['file_name'] = filename

                all_extracted_rows.append(row_features) # Adds the completed dictionary into a master list


    # 6. Convert to data frame and save as CSV
    if len(all_extracted_rows) > 0:
        df_acoustic = pd.DataFrame(all_extracted_rows) # Converts the list into a pandas 2D data frame (spreadsheet)

        # Rearranges column positions by moving participant ID and filename to the front
        cols = ['participant_id', 'file_name'] + [c for c in df_acoustic.columns if c not in ['participant_id', 'file_name']]
        df_acoustic = df_acoustic[cols]

        # Exports and Saves the DataFrame directly to Google Drive as a .csv file without row numbers and prints a final success message
        df_acoustic.to_csv(output_csv_path, index=False)
        print(f"\nSUCCESS: Extracted features for {len(df_acoustic)} WAV files.")
        print(f"Saved CSV at: {output_csv_path}")
        # Potential error messages to target debugging
    else:
        print("WARNING: No .wav files found in directory.")
else:
    print(f"ERROR: Could not find folder path: {wav_input_folder}")

#### Step 3: Acoustic Feature Extraction using `librosa`

This section will focus on extracting time domain, frequency domain, and spectral features using the `librosa` library.

In [ ]:
# 1. Install and Import Required Libraries
import os                             # Used for navigating folders and building file paths
import numpy as np                    # Calculates mean, min, max, and std
import librosa                        # Main acoustic engine to derive features
from google.colab import drive        # Connects Google Drive to Google Colab
import pandas as pd                   # Data frame for CSV file

# 2. Connect to Drive
drive.mount('/content/drive')

# 3. Establish File Paths
# Shows where .wav files are stored
wav_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/depaul/depaul_wav'

# Shows where to store the csv file
output_csv_path = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/depaul/depaul_acoustic_features_librosa.csv'

# 4. Define Feature Extraction Function
def extract_librosa_features(audio_file_path): # takes one single audio file path and extracts time domain and spectral features
    features = {} # Creates an empty dictionary to store the features
    try:
        # Load audio file
        y, sr = librosa.load(audio_file_path, sr=None) # As audio file is being read, y stores amplitude numbers over time, and sr stores how many audio readings were taken per second

        # Time Domain Features
        # Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y=y) # Extracts ZCR
        features['mean_zcr'] = float(np.mean(zcr)) # Calculates average ZCR

        # Root Mean Square
        rms = librosa.feature.rms(y=y) # Extracts RMS
        features['mean_rms'] = float(np.mean(rms)) # Calculates average RMS

        # Spectral Features
        # MFCCs (Flatten 13 values into separate keys mfcc_1 through mfcc_13)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) # Extracts 13 coefficients across time windows and creates 2D array
        mean_mfccs = np.mean(mfccs, axis=1) # Averages each of 13 coefficients to create 1D array
        for i, val in enumerate(mean_mfccs): # Loops through 1D array to create 13 keys for feature dictionary
            features[f'mfcc_{i+1}'] = float(val) # Converts Numpy floats to pandas floats so pd can convert to df

        # Spectral Centroids
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0] # Calculates spectral centroid value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_centroid'] = float(np.mean(spectral_centroids)) # Average spectral centroid value

        # Spectral Spread
        # Librosa used spectral_bandwidth instead of spectral_speed
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0] # Calculates spectral spread value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_bandwidth'] = float(np.mean(spectral_bandwidth)) # Average spectral spread value

        # Spectral Rolloff
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0] # Calculates spectral rolloff value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_rolloff'] = float(np.mean(spectral_rolloff)) # Average spectral rolloff

        # Chroma Features (Flatten 12 values into separate keys chroma_1 through chroma_12)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr) # Extracts 12 values and creates 2D array
        mean_chroma = np.mean(chroma, axis=1) # Averages each of 12 values to create 1D array
        for i, val in enumerate(mean_chroma): # Loops through 1D array to create 12 values for feature dictionary
            features[f'chroma_{i+1}'] = float(val) # Converts Numpy floats to pandas floats so pd can convert to df

    except Exception as e: # Triggers ONLY if erroring when loading or reading .wav file
        print(f"Error extracting features from {audio_file_path}: {e}")

        # Creates a backup dictionary with NaN placeholders so the script doesn't lose track of expected columns
        features = {
            'mean_zcr': np.nan,
            'mean_rms': np.nan,
            'mean_spectral_centroid': np.nan,
            'mean_spectral_bandwidth': np.nan,
            'mean_spectral_rolloff': np.nan
        }
        for i in range(1, 14):
            features[f'mfcc_{i}'] = np.nan
        for i in range(1, 13):
            features[f'chroma_{i}'] = np.nan

    return features

# 5. Feature Extraction Master List
all_extracted_rows = [] # Creates an empty list to capture all extracted features from .wav files

if os.path.exists(wav_input_folder): # Checks if .wav file folder exists
    for current_dir, subdirs, files in os.walk(wav_input_folder): # Checks the main folder and any subfolders inside it
        for filename in files:
            if filename.endswith('.wav'): # Isolates files that end with .wav extension

            # audio_file_path creation
              audio_file_path = os.path.join(current_dir, filename) # Joins folder path and filename into complete file path

print(f"Extracting Librosa features from: {filename}")

row_features = extract_librosa_features(audio_file_path) # Calls the Librosa extraction function from step 4 to compute spectral & time-domain features

# Removes _PAR_clean.wav to create a clean participant ID, and stores the raw filename
row_features['participant_id'] = filename.replace('_PAR_clean.wav', '')
row_features['file_name'] = filename

all_extracted_rows.append(row_features) # Adds the completed dictionary into our master list

    # 6. Convert to DataFrame and Save as CSV
if len(all_extracted_rows) > 0: # Executes when the list is NOT empty and has at least 1 .wav file# Converts the list into a pandas 2D data frame (spreadsheet)
        df_librosa = pd.DataFrame(all_extracted_rows)

        # Rearranges column positions by moving participant ID and filename to the front
        cols = ['participant_id', 'file_name'] + [c for c in df_librosa.columns if c not in ['participant_id', 'file_name']]
        df_librosa = df_librosa[cols]

        # Exports and Saves the DataFrame directly to Google Drive as a .csv file without row numbers and prints a success message
        # Ensure target directory exists before saving
        os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
        df_librosa.to_csv(output_csv_path, index=False)

        print(f"\nSUCCESS: Extracted Librosa features for {len(df_librosa)} WAV files.")
        print(f"Saved CSV at: {output_csv_path}")

else: # Potential error messages to target debugging
        print("WARNING: No .wav files found in directory.")

### Dataset 2: Hopkins
50 .cha files, 47 .mp3 files, 47 .wav files

3 .cha files didn't have a .mp3 file

#### Step 1: Converting `.mp3` files to `.wav` files by using `pydub` library

This is done to ensure consistent processing and compatibility with `parselmouth` and `librosa`.

In [ ]:
# 1. Import required libraries
# Install if needed
import os # Used for navigating folders, creating directories, and finding folders
import re # re stands for Regular Expressions which is a tool that searches for specific patterns (like timestamps) in text files
from google.colab import drive # Lets python read and write files directly inside google drive
from pydub import AudioSegment # AudioSegment is the audio engine in pydub library which converts audio files (.mp3/.mp4 to .wav)

# 2. Connect Drive and Colab so files can be accessed
drive.mount('/content/drive')

# 3. Establishing file paths
# Shows where the .cha and .mp3/.mp4 files are
root_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/hopkins/hopkins_raw'
# Shows where .wav files should be store
output_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/hopkins/hopkins_wav'

os.makedirs(output_folder, exist_ok=True) # Creates the output_folder if it doesn't already exist to prevent code from crashing

# 4. Writing the function to extract millisecond timestamps for *PAR:
def extract_par_timestamps(cha_file_path): # Takes one .cha file at a time
    timestamps = [] # Make an empty list to store the start_time and end_time millisecond pairs
    timestamp_pattern = re.compile(r'[\x15\u0015](\d+)_(\d+)[\x15\u0015]') # Tells Python the search rule to find timestamp numbers in DementiaBank CHAT Transcripts

    with open(cha_file_path, 'r', encoding='utf-8', errors='ignore') as file: # Opens .cha text transcript line by line
        for line in file: # Loops through every line of text in the transcript
            if line.startswith('*PAR:'): # Only process lines where the Participant (*PAR:) is speaking and ignores lines with *INV:
                match = timestamp_pattern.search(line) # Search a specific *PAR: line for timestamps and then move to other *PAR: line
                if match:
                    start_ms = int(match.group(1)) # Converts the start timestamp text into real integer numbers
                    end_ms = int(match.group(2)) # Converts the end timestamp text into real integer numbers
                    timestamps.append((start_ms, end_ms)) # Organizes the timestamps by listing start and end times
    return timestamps # Returns the complete list of patient speaking intervals back to the main program

# 5. Scans folders to find audio and text files
if os.path.exists(root_input_folder): # Checks if depaul_raw (file folder) exists in google drive so code doesn't crash
    found_media_count = 0 # Tracks how many audio files get found - Starts with 0

    for current_dir, subdirs, files in os.walk(root_input_folder): # Tells os.walk (a part of os library) to check the main folder and EVERY subfolder inside it just in case large files need organization and prevents code crash

        file_map = {f.lower(): f for f in files} # Builds a dictionary of file names so Python doesn't get confused by uppercase and lowercase extensions

        for filename in files: # Loops through each file found in the directory

         # Splits the file name into its extension (.mp3) and ID
            ext = os.path.splitext(filename)[1].lower()  # Gets ONLY the extension (.mp3) so ext can trigger file conversion
            base_name = os.path.splitext(filename)[0] # Gets ONLY the patient ID so exact .cha file can be matched


            if ext in ['.mp3', '.mp4', '.wav', '.m4a']: # Checks if file is a recognized audio/video file by utilizing seperated extension
                found_media_count += 1 # found_media_count (0) + 1 counts how many audio/video files it found
                print(f"Processing: {os.path.join(current_dir, filename)}")

                cha_filename = base_name + '.cha' # Seperated Patient ID + .cha to know the full file name of .cha files
                cha_path = os.path.join(current_dir, cha_filename) # Tells python where that specific .cha file is

                if os.path.exists(cha_path): # Confirms if matching .cha file exists
                    print(f"  Found matching .cha file: {cha_path}")
                    timestamps = extract_par_timestamps(cha_path) # Pulls all *PAR: timestamps out of the .cha file according to the function in part 4

                    if timestamps:
                        try: # Opens a safety block so one corrupted audio file won't crash whole code
                            # Load the full audio file into memory with the correct extension as each ext is run because the extensions were seperated
                            if ext == '.mp3':
                                full_audio = AudioSegment.from_mp3(os.path.join(current_dir, filename))
                            elif ext == '.mp4' or ext == '.m4a':
                                full_audio = AudioSegment.from_file(os.path.join(current_dir, filename), format='m4a')
                            elif ext == '.wav':
                                full_audio = AudioSegment.from_wav(os.path.join(current_dir, filename))

                            patient_audio = AudioSegment.empty() # Creates blank audio space where only patient intervals are loaded

                            for start_ms, end_ms in timestamps:  # Concentrates on the timestamps *PAR: is speaking
                            # Ensures timestamps are within audio bounds
                                start_ms = max(0, start_ms)   # Check if audio is below 0
                                end_ms = min(len(full_audio), end_ms) # Check if audio above end time of audio file
                                if start_ms < end_ms: # Safety net to ensure start time is before end time
                                    patient_audio += full_audio[start_ms:end_ms] # Loads the participant speech into the blank audio space

                            if patient_audio.duration_seconds > 0: # Verifies we actually extracted valid audio

                                relative_path = os.path.relpath(current_dir, root_input_folder) # Finds folder branch name of depaul_raw (file folder)
                                output_subdir = os.path.join(output_folder, relative_path) # Glues the branch name onto output path so each wav file in same subfolder name as in the original folder
                                os.makedirs(output_subdir, exist_ok=True) # Creates the output folder if it doesn't already exist so code doesn't crash

                                output_wav_path = os.path.join(output_subdir, f"{base_name}_PAR_clean.wav") # Names the new .wav files
                                patient_audio.export(output_wav_path, format="wav") # Exported into drive

                               # Print helpful messages that convey if code is a success
                               # OR pinpoint where the warning occurred to target debugging at a specific aspect of code that's causing the issue
                                print(f"    SUCCESS: Clean patient WAV saved at: {output_wav_path}")
                            else:
                                print(f"    WARNING: No valid patient speech segments extracted for {filename}.")
                        except Exception as e:
                            print(f"    ERROR processing {filename}: {e}")
                    else:
                        print(f"    WARNING: No *PAR: timestamps found in {cha_filename}.")
                else:
                    print(f"  WARNING: No matching .cha file ({cha_filename}) found in {current_dir}.")

    if found_media_count == 0:
        print(f"No audio files (mp3, mp4, wav, m4a) found in {root_input_folder} or its subfolders.")
else:
    print(f"ERROR: Could not find the specified root input directory: {root_input_folder}")

print("\nBatch processing complete.")



#### Step 2: Acoustic Feature Extraction using `parselmouth`

This section will focus on extracting pitch, voice quality, prosodic, and formant features using the `parselmouth` library.

In [ ]:
# 1. Install and Import Required Libraries
import os                         # Used for navigating folders and building file paths
import numpy as np                # Calculates mean, min, max, and std
import pandas as pd               # Converts results into a clean data frame and exports CSV
from google.colab import drive    # Connects Google Drive to Google Colab
!pip install Praat-Parselmouth==0.4.7 # Install praat-parselmouth
import parselmouth                # Python library for Praat acoustic engine
from parselmouth.praat import call # Executes Praat C++ functions

# 2. Connect to Drive
drive.mount('/content/drive')

# 3. Establish File Path
# Shows where .wav files are stored
wav_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/hopkins/hopkins_wav'

# Shows where to store the csv file
output_csv_path = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/hopkins/hopkins_csv/hopkins_acoustic_features_parselmouth.csv'

# 4. Define feature extraction function
def extract_parselmouth_features(audio_file_path): # takes one single audio file path and extracts pitch, jitter, shimmer, intensity, speaking rate proxy, and formants

    features = {} # Creates an empty dictionary to store the features

    try:
        sound = parselmouth.Sound(audio_file_path) # Loads audio into Praat's memory
        total_duration = sound.get_total_duration() # Measures total duration of audio clip in seconds

        # Part A. Pitch (F0) Features (mean, min, max, std)
        pitch = sound.to_pitch() # Creates Parselmouth pitch object containing calculated frequencies and corresponding time stamps
        pitch_values = pitch.selected_array["frequency"] # Stores Frequency numbers in Hz as 1D Numpy array so it can be used to calculate F0 metrics
        pitch_values = pitch_values[pitch_values != 0] # Filter out silent sound frames because if a person spoke half the time and remained silent for half the time, keeping 0 cuts their average in half

        if len(pitch_values) > 0: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_f0'] = float(np.mean(pitch_values)) # Average
            features['min_f0'] = float(np.min(pitch_values)) # Lowest
            features['max_f0'] = float(np.max(pitch_values)) # Highest
            features['std_f0'] = float(np.std(pitch_values)) # Variation
        else: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_f0'] = 0.0
            features['min_f0'] = 0.0
            features['max_f0'] = 0.0
            features['std_f0'] = 0.0

        # Part B. Voice Quality (Jitter and Shimmer)
        # Calls Praat's C++ to scan sound wave and locate vocal cord pulses between 75 Hz - 500 Hz which is the normal range of Hz
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
        # Asks praat to calculate jitter, multiples by 100 to get %, and saves it
        features['jitter'] = float(call(point_process, "Get jitter (local)", 0.0, 0.0, 0.0001, 0.02, 1.3) * 100.0)
        # Asks praat to calculate shimmer, multiples by 100 to get %, and saves it
        features['shimmer'] = float(call([sound, point_process], "Get shimmer (local)", 0.0, 0.0, 0.0001, 0.02, 1.3, 1.6) * 100.0)

        # Part C. Prosodic Features
        # Intensity Statistics (mean, min, max, std)
        intensity = sound.to_intensity() # Converts audio to intensity object containing sound pressure (loudness) over time
        intensity_values = intensity.values[0] # Extracts volume in decibels for every single step
        valid_intensity = intensity_values[intensity_values > 0] # Filter out zero value frames so silent background gaps don't drag down speaker's real voice volume

        if len(valid_intensity) > 0: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_intensity'] = float(np.mean(valid_intensity)) # Average
            features['min_intensity'] = float(np.min(valid_intensity)) # Lowest
            features['max_intensity'] = float(np.max(valid_intensity)) # Highest
            features['std_intensity'] = float(np.std(valid_intensity)) # Variation
        else: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_intensity'] = 0.0
            features['min_intensity'] = 0.0
            features['max_intensity'] = 0.0
            features['std_intensity'] = 0.0

        # Speaking Rate and Duration
        features['total_duration_sec'] = float(total_duration) # Stores total length of audio file as seconds (duration)

        # Using Praat TextGrid silences
        textgrid = call(sound, "To TextGrid (silences)", 100, 0.0, -25.0, 0.1, 0.1, "silent", "sounding") # Tells Praat to slice audio into intervals labeled either sounding or silent based on decibels drops below -25 dB to isolate active vocalization and pause time, and calculate Articulation Ratio
        num_intervals = call(textgrid, "Get number of intervals", 1) # Asks TextGrid object for total count of alternating speech and pause slices.
        speech_duration = 0.0 # Creates tracker variable starting at 0 to track active talking time

        for i in range(1, num_intervals + 1): # To loop through every interval slice from 1 to the total number of intervals
            label = call(textgrid, "Get label of interval", 1, i) # Checks the text tag of current slice to see if silent or sounding
            if label == "sounding": # If the slice is active speech....
            # Extract the exact start and end timestamp in seconds to know where that speech slice begins and ends.
                start = call(textgrid, "Get start time of interval", 1, i)
                end = call(textgrid, "Get end time of interval", 1, i)
                speech_duration += (end - start) # To get total active speech duration

        features['speech_duration_sec'] = float(speech_duration) # Saves the final duration into the dictionary

        # Speaking Rate Metric (Active Speech / Total Time):
        features['speaking_rate_proxy'] = float(speech_duration / total_duration) if total_duration > 0 else 0.0 # Divides active speech time over duration in seconds to calculate articulation ratio; defaults to 0 if duration is 0

        # Part D. Formants (F1, F2, F3)
        formant = sound.to_formant_burg(time_step=0.01, maximum_formant=5500, max_number_of_formants=5) # Runs Burg's linear prediction algorithm (subtracts the vocal cords vibrations so shape of mouth during speech/formant features can be extracted) to track vocal tract movements up to 5500 Hz every 0.01 seconds.
        f1_vals, f2_vals, f3_vals = [], [], [] # Creates 3 empty lists to store three metrics

        for t in formant.ts(): # Loops through every time point formant measurements were taken
        # Fetches the specific frequency values (in Hz) for Formant 1, Formant 2, and Formant 3 at time t.
            f1 = formant.get_value_at_time(1, t)
            f2 = formant.get_value_at_time(2, t)
            f3 = formant.get_value_at_time(3, t)
# Checks if each value is missing a value (NaN) or is 0 and adds valid numbers to their lists
            if not np.isnan(f1) and f1 > 0: f1_vals.append(f1)
            if not np.isnan(f2) and f2 > 0: f2_vals.append(f2)
            if not np.isnan(f3) and f3 > 0: f3_vals.append(f3)
# Calculates average if more than 0 or sets to 0 if no formants detected
        features['mean_f1'] = float(np.mean(f1_vals)) if f1_vals else 0.0
        features['mean_f2'] = float(np.mean(f2_vals)) if f2_vals else 0.0
        features['mean_f3'] = float(np.mean(f3_vals)) if f3_vals else 0.0

    except Exception as e: # Catches file corruption error, prints error message, makes it 0, and keeps code running
        print(f"Error processing audio file: {e}")
        features = {
            'mean_f0': 0.0, 'min_f0': 0.0, 'max_f0': 0.0, 'std_f0': 0.0,
            'jitter': 0.0, 'shimmer': 0.0,
            'mean_intensity': 0.0, 'min_intensity': 0.0, 'max_intensity': 0.0, 'std_intensity': 0.0,
            'total_duration_sec': 0.0, 'speech_duration_sec': 0.0, 'speaking_rate_proxy': 0.0,
            'mean_f1': 0.0, 'mean_f2': 0.0, 'mean_f3': 0.0
        }

    return features


# 5. Feature Extraction Master List
all_extracted_rows = [] # Creates an empty file to capture all extracted features from .wav file

if os.path.exists(wav_input_folder): # Checks if .wav file folder exists
    for current_dir, subdirs, files in os.walk(wav_input_folder): # Checks the main folder and any subfolders inside it
        for filename in files:
            if filename.endswith('.wav'): # Isolates files that end with .wav extension

                # audio_file_path creation ***
                audio_file_path = os.path.join(current_dir, filename) # Joins the folder path and filename into complete file path string: audio_file_path

                print(f"Extracting features from: {filename}")

                # Calls the main function in step 4 to be executed and extract all acoustic features
                row_features = extract_parselmouth_features(audio_file_path)

                # Removes _PAR_clean.wav to create a clean participant ID, and stores the raw filename
                row_features['participant_id'] = filename.replace('_PAR_clean.wav', '')
                row_features['file_name'] = filename

                all_extracted_rows.append(row_features) # Adds the completed dictionary into a master list


    # 6. Convert to data frame and save as CSV
    if len(all_extracted_rows) > 0:
        df_acoustic = pd.DataFrame(all_extracted_rows) # Converts the list into a pandas 2D data frame (spreadsheet)

        # Rearranges column positions by moving participant ID and filename to the front
        cols = ['participant_id', 'file_name'] + [c for c in df_acoustic.columns if c not in ['participant_id', 'file_name']]
        df_acoustic = df_acoustic[cols]

        # Exports and Saves the DataFrame directly to Google Drive as a .csv file without row numbers and prints a final success message
        df_acoustic.to_csv(output_csv_path, index=False)
        print(f"\nSUCCESS: Extracted features for {len(df_acoustic)} WAV files.")
        print(f"Saved CSV at: {output_csv_path}")
        # Potential error messages to target debugging
    else:
        print("WARNING: No .wav files found in directory.")
else:
    print(f"ERROR: Could not find folder path: {wav_input_folder}")

#### Step 3: Acoustic Feature Extraction using `librosa`

This section will focus on extracting time domain, frequency domain, and spectral features using the `librosa` library.

In [ ]:
# 1. Install and Import Required Libraries
import os                             # Used for navigating folders and building file paths
import numpy as np                    # Calculates mean, min, max, and std
import librosa                        # Main acoustic engine to derive features
from google.colab import drive        # Connects Google Drive to Google Colab
import pandas as pd                   # Data frame for CSV file

# 2. Connect to Drive
drive.mount('/content/drive')

# 3. Establish File Paths
# Shows where .wav files are stored
wav_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/hopkins/hopkins_wav'

# Shows where to store the csv file
output_csv_path = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/hopkins/hopkins_csv/hopkins_acoustic_features_librosa.csv'

# 4. Define Feature Extraction Function
def extract_librosa_features(audio_file_path): # takes one single audio file path and extracts time domain and spectral features
    features = {} # Creates an empty dictionary to store the features
    try:
        # Load audio file
        y, sr = librosa.load(audio_file_path, sr=None) # As audio file is being read, y stores amplitude numbers over time, and sr stores how many audio readings were taken per second

        # Time Domain Features
        # Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y=y) # Extracts ZCR
        features['mean_zcr'] = float(np.mean(zcr)) # Calculates average ZCR

        # Root Mean Square
        rms = librosa.feature.rms(y=y) # Extracts RMS
        features['mean_rms'] = float(np.mean(rms)) # Calculates average RMS

        # Spectral Features
        # MFCCs (Flatten 13 values into separate keys mfcc_1 through mfcc_13)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) # Extracts 13 coefficients across time windows and creates 2D array
        mean_mfccs = np.mean(mfccs, axis=1) # Averages each of 13 coefficients to create 1D array
        for i, val in enumerate(mean_mfccs): # Loops through 1D array to create 13 keys for feature dictionary
            features[f'mfcc_{i+1}'] = float(val) # Converts Numpy floats to pandas floats so pd can convert to df

        # Spectral Centroids
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0] # Calculates spectral centroid value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_centroid'] = float(np.mean(spectral_centroids)) # Average spectral centroid value

        # Spectral Spread
        # Librosa used spectral_bandwidth instead of spectral_speed
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0] # Calculates spectral spread value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_bandwidth'] = float(np.mean(spectral_bandwidth)) # Average spectral spread value

        # Spectral Rolloff
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0] # Calculates spectral rolloff value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_rolloff'] = float(np.mean(spectral_rolloff)) # Average spectral rolloff

        # Chroma Features (Flatten 12 values into separate keys chroma_1 through chroma_12)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr) # Extracts 12 values and creates 2D array
        mean_chroma = np.mean(chroma, axis=1) # Averages each of 12 values to create 1D array
        for i, val in enumerate(mean_chroma): # Loops through 1D array to create 12 values for feature dictionary
            features[f'chroma_{i+1}'] = float(val) # Converts Numpy floats to pandas floats so pd can convert to df

    except Exception as e: # Triggers ONLY if erroring when loading or reading .wav file
        print(f"Error extracting features from {audio_file_path}: {e}")

        # Creates a backup dictionary with NaN placeholders so the script doesn't lose track of expected columns
        features = {
            'mean_zcr': np.nan,
            'mean_rms': np.nan,
            'mean_spectral_centroid': np.nan,
            'mean_spectral_bandwidth': np.nan,
            'mean_spectral_rolloff': np.nan
        }
        for i in range(1, 14):
            features[f'mfcc_{i}'] = np.nan
        for i in range(1, 13):
            features[f'chroma_{i}'] = np.nan

    return features

# 5. Feature Extraction Master List
all_extracted_rows = [] # Creates an empty list to capture all extracted features from .wav files

if os.path.exists(wav_input_folder): # Checks if .wav file folder exists
    for current_dir, subdirs, files in os.walk(wav_input_folder): # Checks the main folder and any subfolders inside it
        for filename in files:
            if filename.endswith('.wav'): # Isolates files that end with .wav extension

                # audio_file_path creation
                audio_file_path = os.path.join(current_dir, filename) # Joins folder path and filename into complete file path

                print(f"Extracting Librosa features from: {filename}")

                # Calls the Librosa extraction function from step 4
                row_features = extract_librosa_features(audio_file_path)

                # Removes _PAR_clean.wav to create a clean participant ID, and stores the raw filename
                row_features['participant_id'] = filename.replace('_PAR_clean.wav', '')
                row_features['file_name'] = filename

                # INDENTED: Adds the completed dictionary to our master list FOR EACH FILE inside the loop
                all_extracted_rows.append(row_features)

# 6. Convert to DataFrame and Save as CSV
if len(all_extracted_rows) > 0: # Executes when the list is NOT empty
    # Converts the list into a pandas 2D data frame
    df_librosa = pd.DataFrame(all_extracted_rows)

    # Rearranges column positions by moving participant ID and filename to the front
    cols = ['participant_id', 'file_name'] + [c for c in df_librosa.columns if c not in ['participant_id', 'file_name']]
    df_librosa = df_librosa[cols]

    # Exports and Saves the DataFrame directly to Google Drive as a .csv file
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df_librosa.to_csv(output_csv_path, index=False)

    print(f"\nSUCCESS: Extracted Librosa features for {len(df_librosa)} WAV files.")
    print(f"Saved CSV at: {output_csv_path}")

else:
    print("WARNING: No .wav files found in directory.")

### Dataset 3: Baycrest
10 .cha files, 10 .mp3 files, 10 .wav files

#### Step 1: Converting `.mp3` files to `.wav` files by using `pydub` library

This is done to ensure consistent processing and compatibility with `parselmouth` and `librosa`.

In [ ]:
# 1. Import required libraries
# Install if needed
import os # Used for navigating folders, creating directories, and finding folders
import re # re stands for Regular Expressions which is a tool that searches for specific patterns (like timestamps) in text files
from google.colab import drive # Lets python read and write files directly inside google drive
from pydub import AudioSegment # AudioSegment is the audio engine in pydub library which converts audio files (.mp3/.mp4 to .wav)

# 2. Connect Drive and Colab so files can be accessed
drive.mount('/content/drive')

# 3. Establishing file paths
# Shows where the .cha and .mp3/.mp4 files are
root_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/baycrest/baycrest_raw'
# Shows where .wav files should be store
output_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/baycrest/baycrest_wav'

os.makedirs(output_folder, exist_ok=True) # Creates the output_folder if it doesn't already exist to prevent code from crashing

# 4. Writing the function to extract millisecond timestamps for *PAR:
def extract_par_timestamps(cha_file_path): # Takes one .cha file at a time
    timestamps = [] # Make an empty list to store the start_time and end_time millisecond pairs
    timestamp_pattern = re.compile(r'[\x15\u0015](\d+)_(\d+)[\x15\u0015]') # Tells Python the search rule to find timestamp numbers in DementiaBank CHAT Transcripts

    with open(cha_file_path, 'r', encoding='utf-8', errors='ignore') as file: # Opens .cha text transcript line by line
        for line in file: # Loops through every line of text in the transcript
            if line.startswith('*PAR:'): # Only process lines where the Participant (*PAR:) is speaking and ignores lines with *INV:
                match = timestamp_pattern.search(line) # Search a specific *PAR: line for timestamps and then move to other *PAR: line
                if match:
                    start_ms = int(match.group(1)) # Converts the start timestamp text into real integer numbers
                    end_ms = int(match.group(2)) # Converts the end timestamp text into real integer numbers
                    timestamps.append((start_ms, end_ms)) # Organizes the timestamps by listing start and end times
    return timestamps # Returns the complete list of patient speaking intervals back to the main program

# 5. Scans folders to find audio and text files
if os.path.exists(root_input_folder): # Checks if depaul_raw (file folder) exists in google drive so code doesn't crash
    found_media_count = 0 # Tracks how many audio files get found - Starts with 0

    for current_dir, subdirs, files in os.walk(root_input_folder): # Tells os.walk (a part of os library) to check the main folder and EVERY subfolder inside it just in case large files need organization and prevents code crash

        file_map = {f.lower(): f for f in files} # Builds a dictionary of file names so Python doesn't get confused by uppercase and lowercase extensions

        for filename in files: # Loops through each file found in the directory

         # Splits the file name into its extension (.mp3) and ID
            ext = os.path.splitext(filename)[1].lower()  # Gets ONLY the extension (.mp3) so ext can trigger file conversion
            base_name = os.path.splitext(filename)[0] # Gets ONLY the patient ID so exact .cha file can be matched


            if ext in ['.mp3', '.mp4', '.wav', '.m4a']: # Checks if file is a recognized audio/video file by utilizing seperated extension
                found_media_count += 1 # found_media_count (0) + 1 counts how many audio/video files it found
                print(f"Processing: {os.path.join(current_dir, filename)}")

                cha_filename = base_name + '.cha' # Seperated Patient ID + .cha to know the full file name of .cha files
                cha_path = os.path.join(current_dir, cha_filename) # Tells python where that specific .cha file is

                if os.path.exists(cha_path): # Confirms if matching .cha file exists
                    print(f"  Found matching .cha file: {cha_path}")
                    timestamps = extract_par_timestamps(cha_path) # Pulls all *PAR: timestamps out of the .cha file according to the function in part 4

                    if timestamps:
                        try: # Opens a safety block so one corrupted audio file won't crash whole code
                            # Load the full audio file into memory with the correct extension as each ext is run because the extensions were seperated
                            if ext == '.mp3':
                                full_audio = AudioSegment.from_mp3(os.path.join(current_dir, filename))
                            elif ext == '.mp4' or ext == '.m4a':
                                full_audio = AudioSegment.from_file(os.path.join(current_dir, filename), format='m4a')
                            elif ext == '.wav':
                                full_audio = AudioSegment.from_wav(os.path.join(current_dir, filename))

                            patient_audio = AudioSegment.empty() # Creates blank audio space where only patient intervals are loaded

                            for start_ms, end_ms in timestamps:  # Concentrates on the timestamps *PAR: is speaking
                            # Ensures timestamps are within audio bounds
                                start_ms = max(0, start_ms)   # Check if audio is below 0
                                end_ms = min(len(full_audio), end_ms) # Check if audio above end time of audio file
                                if start_ms < end_ms: # Safety net to ensure start time is before end time
                                    patient_audio += full_audio[start_ms:end_ms] # Loads the participant speech into the blank audio space

                            if patient_audio.duration_seconds > 0: # Verifies we actually extracted valid audio

                                relative_path = os.path.relpath(current_dir, root_input_folder) # Finds folder branch name of depaul_raw (file folder)
                                output_subdir = os.path.join(output_folder, relative_path) # Glues the branch name onto output path so each wav file in same subfolder name as in the original folder
                                os.makedirs(output_subdir, exist_ok=True) # Creates the output folder if it doesn't already exist so code doesn't crash

                                output_wav_path = os.path.join(output_subdir, f"{base_name}_PAR_clean.wav") # Names the new .wav files
                                patient_audio.export(output_wav_path, format="wav") # Exported into drive

                               # Print helpful messages that convey if code is a success
                               # OR pinpoint where the warning occurred to target debugging at a specific aspect of code that's causing the issue
                                print(f"    SUCCESS: Clean patient WAV saved at: {output_wav_path}")
                            else:
                                print(f"    WARNING: No valid patient speech segments extracted for {filename}.")
                        except Exception as e:
                            print(f"    ERROR processing {filename}: {e}")
                    else:
                        print(f"    WARNING: No *PAR: timestamps found in {cha_filename}.")
                else:
                    print(f"  WARNING: No matching .cha file ({cha_filename}) found in {current_dir}.")

    if found_media_count == 0:
        print(f"No audio files (mp3, mp4, wav, m4a) found in {root_input_folder} or its subfolders.")
else:
    print(f"ERROR: Could not find the specified root input directory: {root_input_folder}")

print("\nBatch processing complete.")



#### Step 2: Acoustic Feature Extraction using `parselmouth`

This section will focus on extracting pitch, voice quality, prosodic, and formant features using the `parselmouth` library.

In [ ]:
# 1. Install and Import Required Libraries
import os                         # Used for navigating folders and building file paths
import numpy as np                # Calculates mean, min, max, and std
import pandas as pd               # Converts results into a clean data frame and exports CSV
from google.colab import drive    # Connects Google Drive to Google Colab
!pip install Praat-Parselmouth==0.4.7 # Install praat-parselmouth
import parselmouth                # Python library for Praat acoustic engine
from parselmouth.praat import call # Executes Praat C++ functions

# 2. Connect to Drive
drive.mount('/content/drive')

# 3. Establish File Path
# Shows where .wav files are stored
wav_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/baycrest/baycrest_wav'

# Shows where to store the csv file
output_csv_path = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/baycrest/baycrest_csv/baycrest_acoustic_features_parselmouth.csv'

# 4. Define feature extraction function
def extract_parselmouth_features(audio_file_path): # takes one single audio file path and extracts pitch, jitter, shimmer, intensity, speaking rate proxy, and formants

    features = {} # Creates an empty dictionary to store the features

    try:
        sound = parselmouth.Sound(audio_file_path) # Loads audio into Praat's memory
        total_duration = sound.get_total_duration() # Measures total duration of audio clip in seconds

        # Part A. Pitch (F0) Features (mean, min, max, std)
        pitch = sound.to_pitch() # Creates Parselmouth pitch object containing calculated frequencies and corresponding time stamps
        pitch_values = pitch.selected_array["frequency"] # Stores Frequency numbers in Hz as 1D Numpy array so it can be used to calculate F0 metrics
        pitch_values = pitch_values[pitch_values != 0] # Filter out silent sound frames because if a person spoke half the time and remained silent for half the time, keeping 0 cuts their average in half

        if len(pitch_values) > 0: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_f0'] = float(np.mean(pitch_values)) # Average
            features['min_f0'] = float(np.min(pitch_values)) # Lowest
            features['max_f0'] = float(np.max(pitch_values)) # Highest
            features['std_f0'] = float(np.std(pitch_values)) # Variation
        else: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_f0'] = 0.0
            features['min_f0'] = 0.0
            features['max_f0'] = 0.0
            features['std_f0'] = 0.0

        # Part B. Voice Quality (Jitter and Shimmer)
        # Calls Praat's C++ to scan sound wave and locate vocal cord pulses between 75 Hz - 500 Hz which is the normal range of Hz
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
        # Asks praat to calculate jitter, multiples by 100 to get %, and saves it
        features['jitter'] = float(call(point_process, "Get jitter (local)", 0.0, 0.0, 0.0001, 0.02, 1.3) * 100.0)
        # Asks praat to calculate shimmer, multiples by 100 to get %, and saves it
        features['shimmer'] = float(call([sound, point_process], "Get shimmer (local)", 0.0, 0.0, 0.0001, 0.02, 1.3, 1.6) * 100.0)

        # Part C. Prosodic Features
        # Intensity Statistics (mean, min, max, std)
        intensity = sound.to_intensity() # Converts audio to intensity object containing sound pressure (loudness) over time
        intensity_values = intensity.values[0] # Extracts volume in decibels for every single step
        valid_intensity = intensity_values[intensity_values > 0] # Filter out zero value frames so silent background gaps don't drag down speaker's real voice volume

        if len(valid_intensity) > 0: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_intensity'] = float(np.mean(valid_intensity)) # Average
            features['min_intensity'] = float(np.min(valid_intensity)) # Lowest
            features['max_intensity'] = float(np.max(valid_intensity)) # Highest
            features['std_intensity'] = float(np.std(valid_intensity)) # Variation
        else: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_intensity'] = 0.0
            features['min_intensity'] = 0.0
            features['max_intensity'] = 0.0
            features['std_intensity'] = 0.0

        # Speaking Rate and Duration
        features['total_duration_sec'] = float(total_duration) # Stores total length of audio file as seconds (duration)

        # Using Praat TextGrid silences
        textgrid = call(sound, "To TextGrid (silences)", 100, 0.0, -25.0, 0.1, 0.1, "silent", "sounding") # Tells Praat to slice audio into intervals labeled either sounding or silent based on decibels drops below -25 dB to isolate active vocalization and pause time, and calculate Articulation Ratio
        num_intervals = call(textgrid, "Get number of intervals", 1) # Asks TextGrid object for total count of alternating speech and pause slices.
        speech_duration = 0.0 # Creates tracker variable starting at 0 to track active talking time

        for i in range(1, num_intervals + 1): # To loop through every interval slice from 1 to the total number of intervals
            label = call(textgrid, "Get label of interval", 1, i) # Checks the text tag of current slice to see if silent or sounding
            if label == "sounding": # If the slice is active speech....
            # Extract the exact start and end timestamp in seconds to know where that speech slice begins and ends.
                start = call(textgrid, "Get start time of interval", 1, i)
                end = call(textgrid, "Get end time of interval", 1, i)
                speech_duration += (end - start) # To get total active speech duration

        features['speech_duration_sec'] = float(speech_duration) # Saves the final duration into the dictionary

        # Speaking Rate Metric (Active Speech / Total Time):
        features['speaking_rate_proxy'] = float(speech_duration / total_duration) if total_duration > 0 else 0.0 # Divides active speech time over duration in seconds to calculate articulation ratio; defaults to 0 if duration is 0

        # Part D. Formants (F1, F2, F3)
        formant = sound.to_formant_burg(time_step=0.01, maximum_formant=5500, max_number_of_formants=5) # Runs Burg's linear prediction algorithm (subtracts the vocal cords vibrations so shape of mouth during speech/formant features can be extracted) to track vocal tract movements up to 5500 Hz every 0.01 seconds.
        f1_vals, f2_vals, f3_vals = [], [], [] # Creates 3 empty lists to store three metrics

        for t in formant.ts(): # Loops through every time point formant measurements were taken
        # Fetches the specific frequency values (in Hz) for Formant 1, Formant 2, and Formant 3 at time t.
            f1 = formant.get_value_at_time(1, t)
            f2 = formant.get_value_at_time(2, t)
            f3 = formant.get_value_at_time(3, t)
# Checks if each value is missing a value (NaN) or is 0 and adds valid numbers to their lists
            if not np.isnan(f1) and f1 > 0: f1_vals.append(f1)
            if not np.isnan(f2) and f2 > 0: f2_vals.append(f2)
            if not np.isnan(f3) and f3 > 0: f3_vals.append(f3)
# Calculates average if more than 0 or sets to 0 if no formants detected
        features['mean_f1'] = float(np.mean(f1_vals)) if f1_vals else 0.0
        features['mean_f2'] = float(np.mean(f2_vals)) if f2_vals else 0.0
        features['mean_f3'] = float(np.mean(f3_vals)) if f3_vals else 0.0

    except Exception as e: # Catches file corruption error, prints error message, makes it 0, and keeps code running
        print(f"Error processing audio file: {e}")
        features = {
            'mean_f0': 0.0, 'min_f0': 0.0, 'max_f0': 0.0, 'std_f0': 0.0,
            'jitter': 0.0, 'shimmer': 0.0,
            'mean_intensity': 0.0, 'min_intensity': 0.0, 'max_intensity': 0.0, 'std_intensity': 0.0,
            'total_duration_sec': 0.0, 'speech_duration_sec': 0.0, 'speaking_rate_proxy': 0.0,
            'mean_f1': 0.0, 'mean_f2': 0.0, 'mean_f3': 0.0
        }

    return features


# 5. Feature Extraction Master List
all_extracted_rows = [] # Creates an empty file to capture all extracted features from .wav file

if os.path.exists(wav_input_folder): # Checks if .wav file folder exists
    for current_dir, subdirs, files in os.walk(wav_input_folder): # Checks the main folder and any subfolders inside it
        for filename in files:
            if filename.endswith('.wav'): # Isolates files that end with .wav extension

                # audio_file_path creation ***
                audio_file_path = os.path.join(current_dir, filename) # Joins the folder path and filename into complete file path string: audio_file_path

                print(f"Extracting features from: {filename}")

                # Calls the main function in step 4 to be executed and extract all acoustic features
                row_features = extract_parselmouth_features(audio_file_path)

                # Removes _PAR_clean.wav to create a clean participant ID, and stores the raw filename
                row_features['participant_id'] = filename.replace('_PAR_clean.wav', '')
                row_features['file_name'] = filename

                all_extracted_rows.append(row_features) # Adds the completed dictionary into a master list


    # 6. Convert to data frame and save as CSV
    if len(all_extracted_rows) > 0:
        df_acoustic = pd.DataFrame(all_extracted_rows) # Converts the list into a pandas 2D data frame (spreadsheet)

        # Rearranges column positions by moving participant ID and filename to the front
        cols = ['participant_id', 'file_name'] + [c for c in df_acoustic.columns if c not in ['participant_id', 'file_name']]
        df_acoustic = df_acoustic[cols]

        # Exports and Saves the DataFrame directly to Google Drive as a .csv file without row numbers and prints a final success message
        df_acoustic.to_csv(output_csv_path, index=False)
        print(f"\nSUCCESS: Extracted features for {len(df_acoustic)} WAV files.")
        print(f"Saved CSV at: {output_csv_path}")
        # Potential error messages to target debugging
    else:
        print("WARNING: No .wav files found in directory.")
else:
    print(f"ERROR: Could not find folder path: {wav_input_folder}")

#### Step 3: Acoustic Feature Extraction using `librosa`

This section will focus on extracting time domain, frequency domain, and spectral features using the `librosa` library.

In [ ]:
# 1. Install and Import Required Libraries
import os                             # Used for navigating folders and building file paths
import numpy as np                    # Calculates mean, min, max, and std
import librosa                        # Main acoustic engine to derive features
from google.colab import drive        # Connects Google Drive to Google Colab
import pandas as pd                   # Data frame for CSV file

# 2. Connect to Drive
drive.mount('/content/drive')

# 3. Establish File Paths
# Shows where .wav files are stored
wav_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/baycrest/baycrest_wav'

# Shows where to store the csv file
output_csv_path = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/baycrest/baycrest_csv/baycrest_acoustic_features_librosa.csv'

# 4. Define Feature Extraction Function
def extract_librosa_features(audio_file_path): # takes one single audio file path and extracts time domain and spectral features
    features = {} # Creates an empty dictionary to store the features
    try:
        # Load audio file
        y, sr = librosa.load(audio_file_path, sr=None) # As audio file is being read, y stores amplitude numbers over time, and sr stores how many audio readings were taken per second

        # Time Domain Features
        # Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y=y) # Extracts ZCR
        features['mean_zcr'] = float(np.mean(zcr)) # Calculates average ZCR

        # Root Mean Square
        rms = librosa.feature.rms(y=y) # Extracts RMS
        features['mean_rms'] = float(np.mean(rms)) # Calculates average RMS

        # Spectral Features
        # MFCCs (Flatten 13 values into separate keys mfcc_1 through mfcc_13)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) # Extracts 13 coefficients across time windows and creates 2D array
        mean_mfccs = np.mean(mfccs, axis=1) # Averages each of 13 coefficients to create 1D array
        for i, val in enumerate(mean_mfccs): # Loops through 1D array to create 13 keys for feature dictionary
            features[f'mfcc_{i+1}'] = float(val) # Converts Numpy floats to pandas floats so pd can convert to df

        # Spectral Centroids
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0] # Calculates spectral centroid value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_centroid'] = float(np.mean(spectral_centroids)) # Average spectral centroid value

        # Spectral Spread
        # Librosa used spectral_bandwidth instead of spectral_speed
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0] # Calculates spectral spread value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_bandwidth'] = float(np.mean(spectral_bandwidth)) # Average spectral spread value

        # Spectral Rolloff
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0] # Calculates spectral rolloff value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_rolloff'] = float(np.mean(spectral_rolloff)) # Average spectral rolloff

        # Chroma Features (Flatten 12 values into separate keys chroma_1 through chroma_12)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr) # Extracts 12 values and creates 2D array
        mean_chroma = np.mean(chroma, axis=1) # Averages each of 12 values to create 1D array
        for i, val in enumerate(mean_chroma): # Loops through 1D array to create 12 values for feature dictionary
            features[f'chroma_{i+1}'] = float(val) # Converts Numpy floats to pandas floats so pd can convert to df

    except Exception as e: # Triggers ONLY if erroring when loading or reading .wav file
        print(f"Error extracting features from {audio_file_path}: {e}")

        # Creates a backup dictionary with NaN placeholders so the script doesn't lose track of expected columns
        features = {
            'mean_zcr': np.nan,
            'mean_rms': np.nan,
            'mean_spectral_centroid': np.nan,
            'mean_spectral_bandwidth': np.nan,
            'mean_spectral_rolloff': np.nan
        }
        for i in range(1, 14):
            features[f'mfcc_{i}'] = np.nan
        for i in range(1, 13):
            features[f'chroma_{i}'] = np.nan

    return features

# 5. Feature Extraction Master List
all_extracted_rows = [] # Creates an empty list to capture all extracted features from .wav files

if os.path.exists(wav_input_folder): # Checks if .wav file folder exists
    for current_dir, subdirs, files in os.walk(wav_input_folder): # Checks the main folder and any subfolders inside it
        for filename in files:
            if filename.endswith('.wav'): # Isolates files that end with .wav extension

                # audio_file_path creation
                audio_file_path = os.path.join(current_dir, filename) # Joins folder path and filename into complete file path

                print(f"Extracting Librosa features from: {filename}")

                # Calls the Librosa extraction function from step 4
                row_features = extract_librosa_features(audio_file_path)

                # Removes _PAR_clean.wav to create a clean participant ID, and stores the raw filename
                row_features['participant_id'] = filename.replace('_PAR_clean.wav', '')
                row_features['file_name'] = filename

                # INDENTED: Adds the completed dictionary to our master list FOR EACH FILE inside the loop
                all_extracted_rows.append(row_features)

# 6. Convert to DataFrame and Save as CSV
if len(all_extracted_rows) > 0: # Executes when the list is NOT empty
    # Converts the list into a pandas 2D data frame
    df_librosa = pd.DataFrame(all_extracted_rows)

    # Rearranges column positions by moving participant ID and filename to the front
    cols = ['participant_id', 'file_name'] + [c for c in df_librosa.columns if c not in ['participant_id', 'file_name']]
    df_librosa = df_librosa[cols]

    # Exports and Saves the DataFrame directly to Google Drive as a .csv file
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df_librosa.to_csv(output_csv_path, index=False)

    print(f"\nSUCCESS: Extracted Librosa features for {len(df_librosa)} WAV files.")
    print(f"Saved CSV at: {output_csv_path}")

else:
    print("WARNING: No .wav files found in directory.")

### Dataset 4: Pitt
243 .cha files, 243 .mp3 files, 243 .wav files


#### Step 1: Converting `.mp3` files to `.wav` files by using `pydub` library

This is done to ensure consistent processing and compatibility with `parselmouth` and `librosa`.

In [ ]:
# 1. Import required libraries
# Install if needed
import os # Used for navigating folders, creating directories, and finding folders
import re # re stands for Regular Expressions which is a tool that searches for specific patterns (like timestamps) in text files
from google.colab import drive # Lets python read and write files directly inside google drive
from pydub import AudioSegment # AudioSegment is the audio engine in pydub library which converts audio files (.mp3/.mp4 to .wav)

# 2. Connect Drive and Colab so files can be accessed
drive.mount('/content/drive')

# 3. Establishing file paths
# Shows where the .cha and .mp3/.mp4 files are
root_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/pitt/pitt_raw'
# Shows where .wav files should be store
output_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/pitt/pitt_wav'

os.makedirs(output_folder, exist_ok=True) # Creates the output_folder if it doesn't already exist to prevent code from crashing

# 4. Writing the function to extract millisecond timestamps for *PAR:
def extract_par_timestamps(cha_file_path): # Takes one .cha file at a time
    timestamps = [] # Make an empty list to store the start_time and end_time millisecond pairs
    timestamp_pattern = re.compile(r'[\x15\u0015](\d+)_(\d+)[\x15\u0015]') # Tells Python the search rule to find timestamp numbers in DementiaBank CHAT Transcripts

    with open(cha_file_path, 'r', encoding='utf-8', errors='ignore') as file: # Opens .cha text transcript line by line
        for line in file: # Loops through every line of text in the transcript
            if line.startswith('*PAR:'): # Only process lines where the Participant (*PAR:) is speaking and ignores lines with *INV:
                match = timestamp_pattern.search(line) # Search a specific *PAR: line for timestamps and then move to other *PAR: line
                if match:
                    start_ms = int(match.group(1)) # Converts the start timestamp text into real integer numbers
                    end_ms = int(match.group(2)) # Converts the end timestamp text into real integer numbers
                    timestamps.append((start_ms, end_ms)) # Organizes the timestamps by listing start and end times
    return timestamps # Returns the complete list of patient speaking intervals back to the main program

# 5. Scans folders to find audio and text files
if os.path.exists(root_input_folder): # Checks if depaul_raw (file folder) exists in google drive so code doesn't crash
    found_media_count = 0 # Tracks how many audio files get found - Starts with 0

    for current_dir, subdirs, files in os.walk(root_input_folder): # Tells os.walk (a part of os library) to check the main folder and EVERY subfolder inside it just in case large files need organization and prevents code crash

        file_map = {f.lower(): f for f in files} # Builds a dictionary of file names so Python doesn't get confused by uppercase and lowercase extensions

        for filename in files: # Loops through each file found in the directory

         # Splits the file name into its extension (.mp3) and ID
            ext = os.path.splitext(filename)[1].lower()  # Gets ONLY the extension (.mp3) so ext can trigger file conversion
            base_name = os.path.splitext(filename)[0] # Gets ONLY the patient ID so exact .cha file can be matched


            if ext in ['.mp3', '.mp4', '.wav', '.m4a']: # Checks if file is a recognized audio/video file by utilizing seperated extension
                found_media_count += 1 # found_media_count (0) + 1 counts how many audio/video files it found
                print(f"Processing: {os.path.join(current_dir, filename)}")

                cha_filename = base_name + '.cha' # Seperated Patient ID + .cha to know the full file name of .cha files
                cha_path = os.path.join(current_dir, cha_filename) # Tells python where that specific .cha file is

                if os.path.exists(cha_path): # Confirms if matching .cha file exists
                    print(f"  Found matching .cha file: {cha_path}")
                    timestamps = extract_par_timestamps(cha_path) # Pulls all *PAR: timestamps out of the .cha file according to the function in part 4

                    if timestamps:
                        try: # Opens a safety block so one corrupted audio file won't crash whole code
                            # Load the full audio file into memory with the correct extension as each ext is run because the extensions were seperated
                            if ext == '.mp3':
                                full_audio = AudioSegment.from_mp3(os.path.join(current_dir, filename))
                            elif ext == '.mp4' or ext == '.m4a':
                                full_audio = AudioSegment.from_file(os.path.join(current_dir, filename), format='m4a')
                            elif ext == '.wav':
                                full_audio = AudioSegment.from_wav(os.path.join(current_dir, filename))

                            patient_audio = AudioSegment.empty() # Creates blank audio space where only patient intervals are loaded

                            for start_ms, end_ms in timestamps:  # Concentrates on the timestamps *PAR: is speaking
                            # Ensures timestamps are within audio bounds
                                start_ms = max(0, start_ms)   # Check if audio is below 0
                                end_ms = min(len(full_audio), end_ms) # Check if audio above end time of audio file
                                if start_ms < end_ms: # Safety net to ensure start time is before end time
                                    patient_audio += full_audio[start_ms:end_ms] # Loads the participant speech into the blank audio space

                            if patient_audio.duration_seconds > 0: # Verifies we actually extracted valid audio

                                relative_path = os.path.relpath(current_dir, root_input_folder) # Finds folder branch name of depaul_raw (file folder)
                                output_subdir = os.path.join(output_folder, relative_path) # Glues the branch name onto output path so each wav file in same subfolder name as in the original folder
                                os.makedirs(output_subdir, exist_ok=True) # Creates the output folder if it doesn't already exist so code doesn't crash

                                output_wav_path = os.path.join(output_subdir, f"{base_name}_PAR_clean.wav") # Names the new .wav files
                                patient_audio.export(output_wav_path, format="wav") # Exported into drive

                               # Print helpful messages that convey if code is a success
                               # OR pinpoint where the warning occurred to target debugging at a specific aspect of code that's causing the issue
                                print(f"    SUCCESS: Clean patient WAV saved at: {output_wav_path}")
                            else:
                                print(f"    WARNING: No valid patient speech segments extracted for {filename}.")
                        except Exception as e:
                            print(f"    ERROR processing {filename}: {e}")
                    else:
                        print(f"    WARNING: No *PAR: timestamps found in {cha_filename}.")
                else:
                    print(f"  WARNING: No matching .cha file ({cha_filename}) found in {current_dir}.")

    if found_media_count == 0:
        print(f"No audio files (mp3, mp4, wav, m4a) found in {root_input_folder} or its subfolders.")
else:
    print(f"ERROR: Could not find the specified root input directory: {root_input_folder}")

print("\nBatch processing complete.")



#### Step 2: Acoustic Feature Extraction using `parselmouth`

This section will focus on extracting pitch, voice quality, prosodic, and formant features using the `parselmouth` library.

In [ ]:
# 1. Install and Import Required Libraries
import os                         # Used for navigating folders and building file paths
import numpy as np                # Calculates mean, min, max, and std
import pandas as pd               # Converts results into a clean data frame and exports CSV
from google.colab import drive    # Connects Google Drive to Google Colab
!pip install Praat-Parselmouth==0.4.7 # Install praat-parselmouth
import parselmouth                # Python library for Praat acoustic engine
from parselmouth.praat import call # Executes Praat C++ functions

# 2. Connect to Drive
drive.mount('/content/drive')

# 3. Establish File Path
# Shows where .wav files are stored
wav_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/pitt/pitt_wav'

# Shows where to store the csv file
output_csv_path = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/pitt/pitt_csv/pitt_acoustic_features_parselmouth.csv'

# 4. Define feature extraction function
def extract_parselmouth_features(audio_file_path): # takes one single audio file path and extracts pitch, jitter, shimmer, intensity, speaking rate proxy, and formants

    features = {} # Creates an empty dictionary to store the features

    try:
        sound = parselmouth.Sound(audio_file_path) # Loads audio into Praat's memory
        total_duration = sound.get_total_duration() # Measures total duration of audio clip in seconds

        # Part A. Pitch (F0) Features (mean, min, max, std)
        pitch = sound.to_pitch() # Creates Parselmouth pitch object containing calculated frequencies and corresponding time stamps
        pitch_values = pitch.selected_array["frequency"] # Stores Frequency numbers in Hz as 1D Numpy array so it can be used to calculate F0 metrics
        pitch_values = pitch_values[pitch_values != 0] # Filter out silent sound frames because if a person spoke half the time and remained silent for half the time, keeping 0 cuts their average in half

        if len(pitch_values) > 0: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_f0'] = float(np.mean(pitch_values)) # Average
            features['min_f0'] = float(np.min(pitch_values)) # Lowest
            features['max_f0'] = float(np.max(pitch_values)) # Highest
            features['std_f0'] = float(np.std(pitch_values)) # Variation
        else: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_f0'] = 0.0
            features['min_f0'] = 0.0
            features['max_f0'] = 0.0
            features['std_f0'] = 0.0

        # Part B. Voice Quality (Jitter and Shimmer)
        # Calls Praat's C++ to scan sound wave and locate vocal cord pulses between 75 Hz - 500 Hz which is the normal range of Hz
        point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
        # Asks praat to calculate jitter, multiples by 100 to get %, and saves it
        features['jitter'] = float(call(point_process, "Get jitter (local)", 0.0, 0.0, 0.0001, 0.02, 1.3) * 100.0)
        # Asks praat to calculate shimmer, multiples by 100 to get %, and saves it
        features['shimmer'] = float(call([sound, point_process], "Get shimmer (local)", 0.0, 0.0, 0.0001, 0.02, 1.3, 1.6) * 100.0)

        # Part C. Prosodic Features
        # Intensity Statistics (mean, min, max, std)
        intensity = sound.to_intensity() # Converts audio to intensity object containing sound pressure (loudness) over time
        intensity_values = intensity.values[0] # Extracts volume in decibels for every single step
        valid_intensity = intensity_values[intensity_values > 0] # Filter out zero value frames so silent background gaps don't drag down speaker's real voice volume

        if len(valid_intensity) > 0: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_intensity'] = float(np.mean(valid_intensity)) # Average
            features['min_intensity'] = float(np.min(valid_intensity)) # Lowest
            features['max_intensity'] = float(np.max(valid_intensity)) # Highest
            features['std_intensity'] = float(np.std(valid_intensity)) # Variation
        else: # Executes when the array is not empty and there is at least 1 valid sound frame
            features['mean_intensity'] = 0.0
            features['min_intensity'] = 0.0
            features['max_intensity'] = 0.0
            features['std_intensity'] = 0.0

        # Speaking Rate and Duration
        features['total_duration_sec'] = float(total_duration) # Stores total length of audio file as seconds (duration)

        # Using Praat TextGrid silences
        textgrid = call(sound, "To TextGrid (silences)", 100, 0.0, -25.0, 0.1, 0.1, "silent", "sounding") # Tells Praat to slice audio into intervals labeled either sounding or silent based on decibels drops below -25 dB to isolate active vocalization and pause time, and calculate Articulation Ratio
        num_intervals = call(textgrid, "Get number of intervals", 1) # Asks TextGrid object for total count of alternating speech and pause slices.
        speech_duration = 0.0 # Creates tracker variable starting at 0 to track active talking time

        for i in range(1, num_intervals + 1): # To loop through every interval slice from 1 to the total number of intervals
            label = call(textgrid, "Get label of interval", 1, i) # Checks the text tag of current slice to see if silent or sounding
            if label == "sounding": # If the slice is active speech....
            # Extract the exact start and end timestamp in seconds to know where that speech slice begins and ends.
                start = call(textgrid, "Get start time of interval", 1, i)
                end = call(textgrid, "Get end time of interval", 1, i)
                speech_duration += (end - start) # To get total active speech duration

        features['speech_duration_sec'] = float(speech_duration) # Saves the final duration into the dictionary

        # Speaking Rate Metric (Active Speech / Total Time):
        features['speaking_rate_proxy'] = float(speech_duration / total_duration) if total_duration > 0 else 0.0 # Divides active speech time over duration in seconds to calculate articulation ratio; defaults to 0 if duration is 0

        # Part D. Formants (F1, F2, F3)
        formant = sound.to_formant_burg(time_step=0.01, maximum_formant=5500, max_number_of_formants=5) # Runs Burg's linear prediction algorithm (subtracts the vocal cords vibrations so shape of mouth during speech/formant features can be extracted) to track vocal tract movements up to 5500 Hz every 0.01 seconds.
        f1_vals, f2_vals, f3_vals = [], [], [] # Creates 3 empty lists to store three metrics

        for t in formant.ts(): # Loops through every time point formant measurements were taken
        # Fetches the specific frequency values (in Hz) for Formant 1, Formant 2, and Formant 3 at time t.
            f1 = formant.get_value_at_time(1, t)
            f2 = formant.get_value_at_time(2, t)
            f3 = formant.get_value_at_time(3, t)
# Checks if each value is missing a value (NaN) or is 0 and adds valid numbers to their lists
            if not np.isnan(f1) and f1 > 0: f1_vals.append(f1)
            if not np.isnan(f2) and f2 > 0: f2_vals.append(f2)
            if not np.isnan(f3) and f3 > 0: f3_vals.append(f3)
# Calculates average if more than 0 or sets to 0 if no formants detected
        features['mean_f1'] = float(np.mean(f1_vals)) if f1_vals else 0.0
        features['mean_f2'] = float(np.mean(f2_vals)) if f2_vals else 0.0
        features['mean_f3'] = float(np.mean(f3_vals)) if f3_vals else 0.0

    except Exception as e: # Catches file corruption error, prints error message, makes it 0, and keeps code running
        print(f"Error processing audio file: {e}")
        features = {
            'mean_f0': 0.0, 'min_f0': 0.0, 'max_f0': 0.0, 'std_f0': 0.0,
            'jitter': 0.0, 'shimmer': 0.0,
            'mean_intensity': 0.0, 'min_intensity': 0.0, 'max_intensity': 0.0, 'std_intensity': 0.0,
            'total_duration_sec': 0.0, 'speech_duration_sec': 0.0, 'speaking_rate_proxy': 0.0,
            'mean_f1': 0.0, 'mean_f2': 0.0, 'mean_f3': 0.0
        }

    return features


# 5. Feature Extraction Master List
all_extracted_rows = [] # Creates an empty file to capture all extracted features from .wav file

if os.path.exists(wav_input_folder): # Checks if .wav file folder exists
    for current_dir, subdirs, files in os.walk(wav_input_folder): # Checks the main folder and any subfolders inside it
        for filename in files:
            if filename.endswith('.wav'): # Isolates files that end with .wav extension

                # audio_file_path creation ***
                audio_file_path = os.path.join(current_dir, filename) # Joins the folder path and filename into complete file path string: audio_file_path

                print(f"Extracting features from: {filename}")

                # Calls the main function in step 4 to be executed and extract all acoustic features
                row_features = extract_parselmouth_features(audio_file_path)

                # Removes _PAR_clean.wav to create a clean participant ID, and stores the raw filename
                row_features['participant_id'] = filename.replace('_PAR_clean.wav', '')
                row_features['file_name'] = filename

                all_extracted_rows.append(row_features) # Adds the completed dictionary into a master list


    # 6. Convert to data frame and save as CSV
    if len(all_extracted_rows) > 0:
        df_acoustic = pd.DataFrame(all_extracted_rows) # Converts the list into a pandas 2D data frame (spreadsheet)

        # Rearranges column positions by moving participant ID and filename to the front
        cols = ['participant_id', 'file_name'] + [c for c in df_acoustic.columns if c not in ['participant_id', 'file_name']]
        df_acoustic = df_acoustic[cols]

        # Exports and Saves the DataFrame directly to Google Drive as a .csv file without row numbers and prints a final success message
        df_acoustic.to_csv(output_csv_path, index=False)
        print(f"\nSUCCESS: Extracted features for {len(df_acoustic)} WAV files.")
        print(f"Saved CSV at: {output_csv_path}")
        # Potential error messages to target debugging
    else:
        print("WARNING: No .wav files found in directory.")
else:
    print(f"ERROR: Could not find folder path: {wav_input_folder}")

#### Step 3: Acoustic Feature Extraction using `librosa`

This section will focus on extracting time domain, frequency domain, and spectral features using the `librosa` library.

In [ ]:
# 1. Install and Import Required Libraries
import os                             # Used for navigating folders and building file paths
import numpy as np                    # Calculates mean, min, max, and std
import librosa                        # Main acoustic engine to derive features
from google.colab import drive        # Connects Google Drive to Google Colab
import pandas as pd                   # Data frame for CSV file

# 2. Connect to Drive
drive.mount('/content/drive')

# 3. Establish File Paths
# Shows where .wav files are stored
wav_input_folder = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/pitt/pitt_wav'

# Shows where to store the csv file
output_csv_path = '/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Data/pitt/pitt_csv/pitt_acoustic_features_librosa.csv'

# 4. Define Feature Extraction Function
def extract_librosa_features(audio_file_path): # takes one single audio file path and extracts time domain and spectral features
    features = {} # Creates an empty dictionary to store the features
    try:
        # Load audio file
        y, sr = librosa.load(audio_file_path, sr=None) # As audio file is being read, y stores amplitude numbers over time, and sr stores how many audio readings were taken per second

        # Time Domain Features
        # Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y=y) # Extracts ZCR
        features['mean_zcr'] = float(np.mean(zcr)) # Calculates average ZCR

        # Root Mean Square
        rms = librosa.feature.rms(y=y) # Extracts RMS
        features['mean_rms'] = float(np.mean(rms)) # Calculates average RMS

        # Spectral Features
        # MFCCs (Flatten 13 values into separate keys mfcc_1 through mfcc_13)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) # Extracts 13 coefficients across time windows and creates 2D array
        mean_mfccs = np.mean(mfccs, axis=1) # Averages each of 13 coefficients to create 1D array
        for i, val in enumerate(mean_mfccs): # Loops through 1D array to create 13 keys for feature dictionary
            features[f'mfcc_{i+1}'] = float(val) # Converts Numpy floats to pandas floats so pd can convert to df

        # Spectral Centroids
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0] # Calculates spectral centroid value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_centroid'] = float(np.mean(spectral_centroids)) # Average spectral centroid value

        # Spectral Spread
        # Librosa used spectral_bandwidth instead of spectral_speed
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0] # Calculates spectral spread value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_bandwidth'] = float(np.mean(spectral_bandwidth)) # Average spectral spread value

        # Spectral Rolloff
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0] # Calculates spectral rolloff value where 0 extracts 1D array frame directly from 2D frame
        features['mean_spectral_rolloff'] = float(np.mean(spectral_rolloff)) # Average spectral rolloff

        # Chroma Features (Flatten 12 values into separate keys chroma_1 through chroma_12)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr) # Extracts 12 values and creates 2D array
        mean_chroma = np.mean(chroma, axis=1) # Averages each of 12 values to create 1D array
        for i, val in enumerate(mean_chroma): # Loops through 1D array to create 12 values for feature dictionary
            features[f'chroma_{i+1}'] = float(val) # Converts Numpy floats to pandas floats so pd can convert to df

    except Exception as e: # Triggers ONLY if erroring when loading or reading .wav file
        print(f"Error extracting features from {audio_file_path}: {e}")

        # Creates a backup dictionary with NaN placeholders so the script doesn't lose track of expected columns
        features = {
            'mean_zcr': np.nan,
            'mean_rms': np.nan,
            'mean_spectral_centroid': np.nan,
            'mean_spectral_bandwidth': np.nan,
            'mean_spectral_rolloff': np.nan
        }
        for i in range(1, 14):
            features[f'mfcc_{i}'] = np.nan
        for i in range(1, 13):
            features[f'chroma_{i}'] = np.nan

    return features

# 5. Feature Extraction Master List
all_extracted_rows = [] # Creates an empty list to capture all extracted features from .wav files

if os.path.exists(wav_input_folder): # Checks if .wav file folder exists
    for current_dir, subdirs, files in os.walk(wav_input_folder): # Checks the main folder and any subfolders inside it
        for filename in files:
            if filename.endswith('.wav'): # Isolates files that end with .wav extension

                # audio_file_path creation
                audio_file_path = os.path.join(current_dir, filename) # Joins folder path and filename into complete file path

                print(f"Extracting Librosa features from: {filename}")

                # Calls the Librosa extraction function from step 4
                row_features = extract_librosa_features(audio_file_path)

                # Removes _PAR_clean.wav to create a clean participant ID, and stores the raw filename
                row_features['participant_id'] = filename.replace('_PAR_clean.wav', '')
                row_features['file_name'] = filename

                # INDENTED: Adds the completed dictionary to our master list FOR EACH FILE inside the loop
                all_extracted_rows.append(row_features)

# 6. Convert to DataFrame and Save as CSV
if len(all_extracted_rows) > 0: # Executes when the list is NOT empty
    # Converts the list into a pandas 2D data frame
    df_librosa = pd.DataFrame(all_extracted_rows)

    # Rearranges column positions by moving participant ID and filename to the front
    cols = ['participant_id', 'file_name'] + [c for c in df_librosa.columns if c not in ['participant_id', 'file_name']]
    df_librosa = df_librosa[cols]

    # Exports and Saves the DataFrame directly to Google Drive as a .csv file
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df_librosa.to_csv(output_csv_path, index=False)

    print(f"\nSUCCESS: Extracted Librosa features for {len(df_librosa)} WAV files.")
    print(f"Saved CSV at: {output_csv_path}")

else:
    print("WARNING: No .wav files found in directory.")